In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [23]:
data_path = "dataset/Electronic_sales_Sep2023-Sep2024.csv"
df = pd.read_csv(data_path)

print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  str    
 3   Loyalty Member     20000 non-null  str    
 4   Product Type       20000 non-null  str    
 5   SKU                20000 non-null  str    
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  str    
 8   Payment Method     20000 non-null  str    
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  str    
 13  Shipping Type      20000 non-null  str    
 14  Add-ons Purchased  15132 non-null  str    
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4), str(9)
m

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [24]:
# Note: Add-ons dihilangkan dulu untuk sementara (nanti coba aja lagi aowkwk)

df = df[df['Order Status'] == 'Completed']
df = df.drop(columns=['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Rating', 'Payment Method','Order Status', 'Add-ons Purchased', 'Add-on Total', 'Shipping Type'])

df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df['YMW'] = df['Purchase Date'].dt.to_period('W')

df.head()

,Product Type,SKU,Total Price,Unit Price,Quantity,Purchase Date,YMW
1,Tablet,SKU1002,741.09,247.03,3,2024-04-20,2024-04-15/2024-04-21
2,Laptop,SKU1005,1855.84,463.96,4,2023-10-17,2023-10-16/2023-10-22
3,Smartphone,SKU1004,3164.76,791.19,4,2024-08-09,2024-08-05/2024-08-11
4,Smartphone,SKU1001,41.50,20.75,2,2024-05-21,2024-05-20/2024-05-26
5,Smartphone,SKU1001,83.00,20.75,4,2024-05-26,2024-05-20/2024-05-26


In [25]:
print("Null Value")
print(df.isnull().sum())

print("\nDuplicated Value")
print(df.duplicated().sum())

Null Value
Product Type     0
SKU              0
Total Price      0
Unit Price       0
Quantity         0
Purchase Date    0
YMW              0
dtype: int64

Duplicated Value
2561


In [26]:
df = df.groupby(['YMW', 'Product Type', 'SKU']).agg(
    Total_Quantity=('Quantity', 'sum'),
    Total_Revenue=('Total Price', 'sum'),
    Average_Unit_Price=('Unit Price', 'mean')
).reset_index()

df = df.sort_values(by=['SKU', 'YMW']).reset_index(drop=True)

print(df.shape[0])
df.head(10)

468


,YMW,Product Type,SKU,Total_Quantity,Total_Revenue,Average_Unit_Price
0,2024-01-01/2024-01-07,Headphones,HDP456,274,98963.32,361.18
1,2024-01-08/2024-01-14,Headphones,HDP456,252,91017.36,361.18
2,2024-01-15/2024-01-21,Headphones,HDP456,195,70430.10,361.18
3,2024-01-22/2024-01-28,Headphones,HDP456,211,76208.98,361.18
4,2024-01-29/2024-02-04,Headphones,HDP456,193,69707.74,361.18
5,2024-02-05/2024-02-11,Headphones,HDP456,198,71513.64,361.18
6,2024-02-12/2024-02-18,Headphones,HDP456,106,38285.08,361.18
7,2024-02-19/2024-02-25,Headphones,HDP456,221,79820.78,361.18
8,2024-02-26/2024-03-03,Headphones,HDP456,124,44786.32,361.18
9,2024-03-04/2024-03-10,Headphones,HDP456,171,61761.78,361.18


In [27]:
df['Year'] = df['YMW'].dt.year
df['Week'] = df['YMW'].dt.week

df['Qty_1_Minggu_Lalu'] = df.groupby('SKU')['Total_Quantity'].shift(1)
df['Qty_2_Minggu_Lalu'] = df.groupby('SKU')['Total_Quantity'].shift(2)

df = df.dropna().copy()

df = pd.get_dummies(df, columns=['Product Type', 'SKU'], drop_first=True)

print(df.shape[0])
df.head()

448


,YMW,Total_Quantity,Total_Revenue,Average_Unit_Price,Year,Week,Qty_1_Minggu_Lalu,Qty_2_Minggu_Lalu,Product Type_Laptop,Product Type_Smartphone,...,Product Type_Tablet,SKU_LTP123,SKU_SKU1001,SKU_SKU1002,SKU_SKU1003,SKU_SKU1004,SKU_SKU1005,SKU_SMP234,SKU_SWT567,SKU_TBL345
2,2024-01-15/2024-01-21,195,70430.10,361.18,2024,3,252.0,274.0,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2024-01-22/2024-01-28,211,76208.98,361.18,2024,4,195.0,252.0,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2024-01-29/2024-02-04,193,69707.74,361.18,2024,5,211.0,195.0,False,False,...,False,False,False,False,False,False,False,False,False,False
5,2024-02-05/2024-02-11,198,71513.64,361.18,2024,6,193.0,211.0,False,False,...,False,False,False,False,False,False,False,False,False,False
6,2024-02-12/2024-02-18,106,38285.08,361.18,2024,7,198.0,193.0,False,False,...,False,False,False,False,False,False,False,False,False,False


In [28]:
unique_week = sorted(df['YMW'].unique())
test_time_limit = unique_week[-4]

train_data = df[df['YMW'] < test_time_limit]
test_data = df[df['YMW'] >= test_time_limit]

y_train = train_data['Total_Quantity']
X_train = train_data.drop(columns=['YMW', 'Total_Quantity', 'Total_Revenue'])

y_test = test_data['Total_Quantity']
X_test = test_data.drop(columns=['YMW', 'Total_Quantity', 'Total_Revenue'])

model = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42)
model.fit(X_train, y_train)

prediction_test = model.predict(X_test)
error = mean_absolute_error(y_test, prediction_test)

print(f"Rata-rata tebakan model meleset sebanyak: {error:.2f} unit produk")

print("\n=== INFO JUMLAH BARIS ===")
print(f"Data Training (Masa Lalu): {X_train.shape[0]} baris")
print(f"Data Testing (Soal Ujian): {X_test.shape[0]} baris")
print(f"Total Baris Keseluruhan: {df.shape[0]} baris\n")

rata_rata_asli = y_test.mean()
print(f"Rata-rata penjualan asli per minggu: {rata_rata_asli:.2f} unit")
print(f"Persentase Error: {(error / rata_rata_asli) * 100:.2f}%")

Rata-rata tebakan model meleset sebanyak: 57.49 unit produk

=== INFO JUMLAH BARIS ===
Data Training (Masa Lalu): 408 baris
Data Testing (Soal Ujian): 40 baris
Total Baris Keseluruhan: 448 baris

Rata-rata penjualan asli per minggu: 136.10 unit
Persentase Error: 42.24%
